### Install CLIP

In [ ]:
!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-q8sf3gd0
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-q8sf3gd0
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=bd3e54721f4e9b29bdc0a56c7bfb029ed2288a1f7f1af0c382c7b837b7520601
  Stored in directory: /tmp/pip-ephem-wheel-cache-265kopi0/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Import the packages

In [ ]:
import numpy as np
import pandas as pd
import clip
import torch
from tqdm import tqdm
from PIL import Image

Load the CLIP model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device = device)

100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 136MiB/s]


In [ ]:
device

'cuda'

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/product_search_engine'
DATA_DIR = f"{PROJECT_DIR}/data"

In [ ]:
import os

!ls /content/drive/MyDrive/product_search_engine

os.listdir(DATA_DIR)

data  embeddings  images  indexes  models  notebooks  src


['styles.csv',
 'images2',
 'fashion-product-images-small.zip',
 'myntradataset',
 'images',
 'cleaned_products.csv',
 'text_processed.csv',
 'image_validated.csv']

In [ ]:
df = pd.read_csv(f"{DATA_DIR}/image_validated.csv")

## Text Embeddings

In [ ]:
text_embeddings = []
texts = df['text_input'].tolist()

# Iterate through the texts in batches of 32
with torch.no_grad():
  for i in tqdm(range(0, len(texts), 32)):
    batch = texts[i : i+32] # Slice of 32 texts for each iteration
    # Convert the batch of text strings into numerical tokens
    # and move these tokens to the GPU for processing
    tokens = clip.tokenize(batch).to(device)
    # Process the tokens and generate dense vector representations (embeddings) for each
    # text in the batch by passing them to the CLIP model
    emb = model.encode_text(tokens)
    # append the generated embeddings to the list storing all the embeddings
    text_embeddings.append(emb)



100%|██████████| 1389/1389 [00:31<00:00, 43.83it/s]


## Image Embeddings

In [ ]:
image_embeddings = []

In [ ]:
# Iterate through each file path
with torch.no_grad():
  for path in tqdm(df['images_path']):
    # CLIP expects the input in batch format, we need to add an extra dimension at the beginning of the tensor
    # Preprocess (resizing, cropping, and normalizing pixel values) the image after converting into a batch
    image = preprocess(Image.open(path)).unsqueeze(0).to(device)
    # Capture the visual features of the image by prepapring the
    # processed image tensor into dense vector representation
    emb = model.encode_image(image)
    image_embeddings.append(emb.cpu())

# Concatenate all the image embeddings into a single tensor
# Useful for sbsequent operations like similarity search, or clustering
image_embeddings = torch.cat(image_embeddings)

100%|██████████| 44419/44419 [4:34:34<00:00,  2.70it/s]


#### Save the embeddings

In [ ]:
np.save(f"{PROJECT_DIR}/embeddings/text_embeddings.npy", torch.cat(text_embeddings).cpu().numpy())
np.save(f"{PROJECT_DIR}/embeddings/image_embeddings.npy", image_embeddings.numpy())